# Lecția 18: Securizarea Agenților AI cu Chitanțe Criptografice

## Caiet Practic

Acest caiet parcurge patru exerciții:

1. **Semnează prima ta chitanță** pentru un apel de unelte agent și verifică-o.
2. **Modifică chitanța** și observă cum eșuează verificarea.
3. **Construiește un lanț de trei chitanțe** și confirmă integritatea lanțului.
4. **Învelește un apel de unealtă Microsoft Agent Framework** astfel încât fiecare acțiune să emită o chitanță.

Toate primitivele criptografice sunt importate din biblioteci bine întreținute (`pynacl` pentru Ed25519, `jcs` pentru JSON canonic RFC 8785, `hashlib` din biblioteca standard Python pentru SHA-256). Logica chitanței în sine este Python simplu pe care îl poți citi și modifica.

Rulează celulele în ordine. Fiecare secțiune este scurtă și independentă.


## Configurare

Instalează cele două dependențe. Ambele au licențe permisive (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Utilitare Ajutătoare

Acești doi ajutători se ocupă de codificarea base64url (fără completare) și de hashing SHA-256 al obiectelor arbitrare. Ei mențin restul caietului concentrat pe logica chitanței propriu-zise.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Secțiunea 1: Semnează prima ta chitanță

Imaginează-ți că agentul nostru pentru **Contoso Travel** tocmai a căutat zboruri de la Sydney la Los Angeles pentru un client. Dorim să înregistrăm declarația gateway-ului despre această apelare a instrumentului astfel încât un auditor viitor să o poată verifica folosind o cheie de emițător atașată separat.

### Pasul 1.1: Generează o cheie de semnare

În producție, cheia de semnare a agentului ar fi ținută într-un modul hardware de securitate (HSM), Azure Key Vault sau un depozit protejat similar. Pentru această lecție generăm o cheie nouă în memorie.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Pasul 1.2: Construiește payload-ul chitanței

Payload-ul conține tot ceea ce vrem ca chitanța să ateste: cine a acționat, ce instrument, cu ce argumente, ce s-a întors, sub ce politică și când. Hash-uim argumentele și rezultatul în loc să le includem inline, astfel încât chitanța să nu divulge conținut sensibil.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Pasul 1.3: Semnează și asamblează chitanța

Trei pași:

1. Canonicalizează payload-ul folosind JCS astfel încât două implementări care produc aceeași chitanță logică să producă biți identici.
2. Semnează biții canonici direct cu cheia privată Ed25519. PureEdDSA face intern hash pe mesaj, deci un pre-hash suplimentar ar schimba protocolul.

Semnătura este apoi atașată payload-ului original pentru a produce chitanța finală.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Pasul 1.4: Verifică chitanța

Verificarea inversează procesul. Eliminăm semnătura, recalculăm octeții canonici și verificăm semnătura în raport cu cheia publică a emitentului.

Verificarea se face offline: nu se apelează niciun serviciu și nu este nevoie de acces la rețea. Totuși, este necesar un singur lucru în afară de chitanță: cheia publică a emitentului, obținută printr-un canal pe care auditorul îl consideră deja de încredere, cum ar fi un Set JWK publicat de emitent sau o valoare fixată anterior. Câmpul `public_key` din interiorul chitanței nu poate juca acest rol, deoarece oricine scrie chitanța scrie și acel câmp. Secțiunea 2 arată falsificarea pe care acesta o previne.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Ar trebui să vedeți `Receipt is valid: True`, `Pre-hashed receipt valid: False` și `Malformed receipts rejected: True`. Cazul pozitiv demonstrează că calea de semnare directă-JCS funcționează; controlul negativ face ca regula domeniului semnăturii să fie executabilă în loc să rămână ca text explicativ; ultima linie arată că o intrare formatată greșit returnează `False` în loc să ridice o excepție.


## Secțiunea 2: Manipularea chitanței

Scopul principal al chitanțelor este să fie evidentă manipularea lor. Să demonstrăm acest lucru.

Vom modifica exact un caracter din chitanță și vom urmări cum verificarea eșuează.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Ce tocmai s-a întâmplat?

Când am schimbat `policy_id`, octeții canonici s-au schimbat. Semnătura (care era peste octeții canonici originali) nu mai corespunde. Verificarea returnează corect `False`.

Nu există nicio modalitate de a schimba un câmp semnat, adică orice în afara obiectului `signature`, și totuși să verifici chitanța împotriva cheii publice a emițătorului, decât dacă atacatorul are cheia privată a emițătorului. Obiectul `signature` însuși nu este semnat: `sig` este semnătura, iar `verify_receipt` ignoră `public_key`, deci editarea doar a acelui câmp nu schimbă nimic. Atâta timp cât cheia privată este într-un seif de chei și verificatorii folosesc cheia publicată independent a emițătorului, schimbarea unui câmp semnat este imposibil de ascuns. Această condiție contează: celula următoare arată o chitanță modificată care se verifică perfect împotriva unei chei diferite.

Încearcă singur: modifică `tool_name` sau `agent_id` sau `timestamp` din celula de mai sus și rulează din nou. Fiecare modificare a unui câmp semnat produce o chitanță invalidă.


### Manipularea pe care semnătura nu o prinde

Atacatorul de mai sus nu a re-semnăt. Unul mai atent ar face asta: ar modifica chitanța,
ar semna-o cu o cheie pe care și-ar fi generat-o singur, și ar pune acea cheie în
`public_key` câmp. Rezultatul este o chitanță a cărei semnătură este complet autentică
pentru conținutul său.

Rulează asta și uită-te care dintre cele două verificări o prinde.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Ar trebui să vedeți `True` apoi `False`. Cele trei linii `assert` fac din asta o
verificare: celula generează o eroare dacă falsul vreodată se verifică față de
cheia emitentului.

Falsul este intern consecvent. Semnătura sa chiar corespunde conținutului său
și cheii pe care o poartă, așa că un verificator care citește cheia din
chitanță îl raportează ca valid. Nimic din chitanță nu pare greșit, pentru că
nimic din ea nu este inconsistent.

Singurul lucru care îl detectează este verificarea față de o cheie pe care atacatorul nu
a avut în vreun fel posibilitatea să o influențeze. Acesta este motivul pentru care `verify_receipt` primește cheia ca argument: o
funcție cu semnătura `verify_receipt(receipt)` nu poate fi scrisă în siguranță,
pentru că singura cheie la care ar putea ajunge este cea furnizată de atacator.

Aceeași chitanță este salvată ca `sample_receipts/04_resigned_receipt.json` dacă doriți
să o inspectați în afara caietului.


## Secțiunea 3: Legați chitanțele între ele

O singură chitanță protejează o acțiune. Majoritatea agenților efectuează multe acțiuni. Pentru a face întreaga secvență evident nealterabilă, legăm fiecare chitanță de cea precedentă prin includerea hash-ului chitanței anterioare în payload-ul noii chitanțe.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Dacă oricine elimină sau rearanjează o chitanță în secvența prezentată, următoarea legătură se rupe. Detectarea trunchierii sau rescrierii sufixului controlate de semnatar necesită, de asemenea, un cap actual fixat, un punct de control extern sau un angajament autentificat al populației.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Acum întrerupe lanțul prin modificarea chitanței din mijloc și reverifică. Chitanța modificată nu trece verificarea semnăturii, ȘI chitanța următoare nu trece verificarea lanțului (deoarece `previous_receipt_hash` nu mai corespunde cu hash-ul chitanței modificate din mijloc).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Chitanța 0 încă se verifică (nu a fost modificată și nu are un predecesor de care să depindă). Chitanța 1 eșuează la verificarea semnăturii pentru că am schimbat `tool_args_hash`. Chitanța 2 eșuează la verificarea lanțului pentru că `previous_receipt_hash` a fost calculat față de chitanța originală (acum modificată) 1.

Chiar dacă un atacator resignează chitanța 1 modificată, noua semnătură eșuează față de cheia emitentului, decât dacă deține cheia privată a emitentului (o resiguranță cu propria lor cheie eșuează, așa cum a arătat Secțiunea 2), iar nepotrivirea lanțului în chitanța 2 ar expune totuși modificarea. Pentru a ascunde schimbarea, atacatorul ar trebui să resigneze fiecare chitanță din punctul modificării încolo, ceea ce necesită posesia cheii private a emitentului.


## Secțiunea 4: Împachetarea unui apel de instrument agent cu semnarea chitanței

Într-o implementare reală, nu doriți ca fiecare autor de agent să își amintească să apeleze `make_receipt`. Doriți ca semnarea chitanței să fie automată pentru fiecare invocare de instrument.

Iată cel mai simplu model: o clasă wrapper care primește orice funcție apelabilă de instrument și returnează o versiune care emite chitanța. Acest model se adaptează la orice cadru agent, inclusiv Microsoft Agent Framework (`agent_framework.foundry`).

Dacă nu aveți configurat un proiect Microsoft Foundry, mock-ul local de mai jos demonstrează încă modelul.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integrarea cu Microsoft Agent Framework

Wrapper-ul `ReceiptedTool` de mai sus este independent de framework. Pentru a-l folosi într-un agent construit cu Microsoft Agent Framework, înregistrează funcția învelită ca un instrument. O schiță (ar trebui să înlocuiești mock-ul cu o înregistrare reală a unui instrument Microsoft Foundry):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Framework-ul agentului nu trebuie să știe nimic despre chitanțe. Semnarea chitanțelor este încapsulată în jurul instrumentului, nu integrată în framework. Așa adaugi proveniență codului existent al agentului fără a rescrie agentul.


## Recap and stretch challenge

You have:

- Generated an Ed25519 key pair.
- Built and signed a receipt for an agent tool call.
- Verified the receipt offline using only the issuer's public key, obtained independently of the receipt.
- Tampered with a receipt and observed verification fail.
- Built a hash-chained sequence of three receipts.
- Tampered with the middle of the chain and observed both signature failure and chain-link failure.
- Wrapped a tool function with automatic receipt signing.

**Stretch challenge.** Extend the receipt schema with a `request_id` field (a UUID for distributed tracing). Update `make_receipt` to include it, and confirm receipts still verify end to end. Then modify the field after signing and confirm verification fails. This forces you to internalize how every byte of the canonical encoding contributes to the signature.

**Important boundary.** Under a pinned key and chain context, receipts prove signature attribution, content integrity, and a link to the presented predecessor. They do NOT prove execution, source truth, trusted time, completeness, policy evaluation, or correctness. Receipts are a foundation. Governance is the system you build on top.

Read the lesson README again with that boundary in mind. The most common mistake teams make with receipts is assuming "we have receipts" means "we are governed." It does not. Receipts make agent behavior auditable. They do not make it correct.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
